In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

print("Libraries imported successfully!")
print("TensorFlow Version:", tf.__version__)

In [ ]:
data = pd.read_csv(
    r"C:\Users\lenovo\OneDrive\Documents\Desktop\experiment 12\nifty_fifty\NSE_Daily_indices_data.csv"
)

print("Dataset loaded successfully!")
print("Original Dataset Shape:", data.shape)

In [ ]:
print("First five rows:")
display(data.head())

print("\nDataset columns:")
print(data.columns.tolist())

print("\nDataset information:")
data.info()

In [ ]:
# Remove extra spaces from column names
data.columns = data.columns.str.strip()

# Filter only NIFTY-50 records
data["SYMBOL"] = data["SYMBOL"].astype(str).str.strip()

data = data[data["SYMBOL"] == "NIFTY"].copy()

print("NIFTY-50 records selected successfully!")
print("NIFTY-50 Dataset Shape:", data.shape)

display(data.head())

In [ ]:
# Select required columns
data = data[
    ["DATETIME", "OPEN", "HIGH", "LOW", "CLOSE"]
].copy()

# Rename columns
data.columns = [
    "Date", "Open", "High", "Low", "Close"
]

# Convert date into datetime format
data["Date"] = pd.to_datetime(
    data["Date"],
    dayfirst=True,
    errors="coerce"
)

# Convert numerical columns
for col in ["Open", "High", "Low", "Close"]:
    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

# Remove missing values
data = data.dropna()

# Sort records chronologically
data = data.sort_values(
    "Date"
).reset_index(drop=True)

print("Data preprocessing completed!")
print("Cleaned Dataset Shape:", data.shape)

display(data.head())

In [ ]:
# Create previous day's closing price
data["Previous_Close"] = data["Close"].shift(1)

# Remove first row containing missing previous close
data = data.dropna().reset_index(drop=True)

print("Previous closing price created successfully!")

display(data.head())

print("\nFinal Dataset Shape:", data.shape)

In [ ]:
# Select input features
X = data[
    ["Open", "High", "Low", "Previous_Close"]
].values

# Select target variable
y = data[["Close"]].values

print("Input Features Shape:", X.shape)
print("Target Shape:", y.shape)

In [ ]:
# Calculate split index
split = int(len(data) * 0.8)

# Training data
X_train = X[:split]
y_train = y[:split]

# Testing data
X_test = X[split:]
y_test = y[split:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("Training input:", X_train.shape)
print("Testing input:", X_test.shape)

In [ ]:
# Initialize scalers
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# Scale input features
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# Scale target values
y_train_scaled = scaler_y.fit_transform(y_train)

print("Feature scaling completed!")

print("Scaled Training Shape:", X_train_scaled.shape)
print("Scaled Testing Shape:", X_test_scaled.shape)

In [ ]:
# Reshape training data
X_train_scaled = X_train_scaled.reshape(
    X_train_scaled.shape[0], 1, 4
)

# Reshape testing data
X_test_scaled = X_test_scaled.reshape(
    X_test_scaled.shape[0], 1, 4
)

print("Data reshaped for LSTM successfully!")

print("Training LSTM Input Shape:", X_train_scaled.shape)
print("Testing LSTM Input Shape:", X_test_scaled.shape)

In [ ]:
# Build LSTM model
model = Sequential()

# LSTM layer with 64 units
model.add(
    LSTM(64, input_shape=(1, 4))
)

# Dense hidden layer
model.add(
    Dense(32, activation="relu")
)

# Output layer
model.add(Dense(1))

# Compile model
model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)

# Display model summary
model.summary()

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train_scaled,
    epochs=50,
    batch_size=32,
    verbose=1
)

print("Model training completed successfully!")

In [ ]:
# Generate predictions
predicted_scaled = model.predict(X_test_scaled)

# Convert predictions to original price scale
predicted_prices = scaler_y.inverse_transform(
    predicted_scaled
)

# Actual closing prices
actual_prices = y_test

print("Prediction completed successfully!")

In [ ]:
results = pd.DataFrame({
    "Actual Closing Price": actual_prices.flatten(),
    "Predicted Closing Price": predicted_prices.flatten()
})

print("Actual vs Predicted Closing Prices:")
display(results.head(10))

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.title("LSTM Training Loss")
plt.xlabel("Epochs")
plt.ylabel("Mean Squared Error")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    actual_prices,
    label="Actual Closing Price",
    linewidth=2
)

plt.plot(
    predicted_prices,
    label="Predicted Closing Price",
    linewidth=2
)

plt.title("NIFTY-50 Actual vs Predicted Closing Prices")

plt.xlabel("Trading Days")
plt.ylabel("Closing Price")

plt.legend()
plt.grid(True)
plt.tight_layout()

plt.show()

In [ ]:
print("========================================")
print(" NIFTY-50 TIME SERIES FORECASTING")
print("========================================")

print("Original Dataset Records: 114419")
print("NIFTY-50 Records:", 7000)
print("Final Cleaned Records:", len(data))

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

print("Model Parameters:", model.count_params())

print("\nExperiment Completed Successfully!")